In [ ]:
import sys
from pathlib import Path

import numpy as np
from tabulate import tabulate
from scipy.interpolate import interp1d

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stepsic.parameters import CosmoParameters
from stepsic.data import CosmoData

import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.WARNING)

$$
    \omega_b = \frac{\omega_b h^{2}}{h^{2}} \qquad\text{with}\quad h=\tfrac{H_0}{100}.
$$

with propagated errors as

$$
    \sigma_{\omega_b}^2
    \;=\;
    \left[ \left(\frac{\sigma_{\omega_b h^2}}{\omega_b h^2}\right)^{\!2}
    \;+\;
    \left(2\,\frac{\sigma_{H_0}}{H_0}\right)^{\!2}\; \right] \rule{0pt}{14pt}\omega_b^{\;2}\,.
$$

In [ ]:
def calculate_omx(omxh2, omxh2_err, H0, H0_err, name=None):
    h = H0 / 100
    omx = omxh2 / h**2
    omx_err = np.sqrt(((omxh2_err / omxh2)**2 + (2*H0_err / H0)**2) * omx**2)
    print(f'{name}:\t{omx:.8f} +- {omx_err:.8f}')
    return omx, omx_err


In [ ]:
# 2.17
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022383, omxh2_err=0.0, H0=67.32, H0_err=0.0, name='best fit 2.17')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02237, omxh2_err=0.00015, H0=67.36, H0_err=0.54, name=r'68% limit 2.17')
# 2.18
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022447, omxh2_err=0.0, H0=67.702, H0_err=0.0, name='best fit 2.18')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02242, omxh2_err=0.00014, H0=67.66, H0_err=0.42, name=r'68% limit 2.18')
# 2.19
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022408, omxh2_err=0.0, H0=67.49, H0_err=0.0, name='best fit 2.19')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02239, omxh2_err=0.00014, H0=67.48, H0_err=0.50, name=r'68% limit 2.19')
# 2.20
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022436, omxh2_err=0.0, H0=67.742, H0_err=0.0, name='best fit 2.20')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02243, omxh2_err=0.00013, H0=67.72, H0_err=0.40, name=r'68% limit 2.20')

## Resolution-Mass Map

In [ ]:
path = Path.cwd().parent / 'config.toml'
params = CosmoParameters(path=path).get_parameters()

In [ ]:
ic = CosmoData.load_snapshot(Path(params['INPUT_GLASS']))
ic.to_internal_units(params)
ic.rescale_snapshot_mass(params)
ic.periodic_shift(params)

### Original

In [ ]:
res = np.cbrt(ic.M_box/ic.mass_list).astype(int)
res_mass_map = np.c_[ic.mass_list, res]
res_tab = np.zeros(params['NGRIDSAMPLES'], dtype=int)
mass_tab = np.zeros(params['NGRIDSAMPLES'], dtype=int)

delta_Nsample = np.ceil(ic.mass_list.size/params['NGRIDSAMPLES']).astype(int)

print("The generated Nsample list:")
print("ID\tNsample\tMass(in 10e11Msol)")
res_tab[-1] = res_mass_map[0, 1]
print("%i\t%i\t%d" % (len(res_tab)-1, res_tab[-1], res_mass_map[0, 0]))
for i in range(len(res_tab)-2, -1, -1):
    mass_tab[i] = res_mass_map[-1-i*delta_Nsample, 0]
    res_tab[i] = res_mass_map[-1-i*delta_Nsample, 1]
    print("%i\t%i\t%d" % (i, res_tab[i], mass_tab[i]))

### Reworked

In [ ]:
def create_nres_mass_map(n_grid_samples, mass_list, M_box, Lbox):
    '''
    Creates a lookup table for the number of voxels per mass bin
    for a variable resolution grid in a regular StePS simulation.

    Parameters
    ----------
    n_grid_samples : int
        Number of grids with different resolutions.
    mass_list : ndarray
        Array containing the sorted unique particle masses.
    M_box : float
        Total mass in the simulation box (in 1e11 Msol).
    Lbox : ndarray
        Box dimensions in [Mpc]. Can be a single scalar for a cubical box or
        an array in the form of `[Lx, Ly, Lz]` for a rectangular cuboid.

    Returns
    -------
    nres_tab : ndarray of shape (n_grid_samples,)
        Array containing the number of resolution elements for each grid.
    mass_tab : ndarray of shape (n_grid_samples,)
        Array containing the mass values corresponding to each grid.
    '''
    nres_list = np.min(Lbox) // np.cbrt(np.prod(Lbox) * mass_list / M_box)
    idx = np.linspace(
        0, mass_list.size - 1, n_grid_samples, endpoint=True, dtype=int)
    
    # Populate lookup table by starting with the outermost mass bin
    nres_tab = nres_list[idx[::-1]]
    mass_tab = mass_list[idx[::-1]]

    log.info('The generated resolution-mass map:')
    print(tabulate([*zip(nres_tab, mass_tab)],
                   headers=['Resolution', 'Mass [1e11Msol]'], floatfmt='.0f'))
    return nres_tab, mass_tab

In [ ]:
nres_tab, mass_tab = create_nres_mass_map(
    params['NGRIDSAMPLES'], ic.mass_list, ic.M_box, params['LBOX'])

In [ ]:
def test_interp():
    disp = np.random.random(params['NGRIDSAMPLES'])
    disp_interp = np.interp(ic.mass, mass_tab, disp)
    return any(disp_interp == disp[-1])
print('Interpolation always returns the highest res displacement:')
any([test_interp() for _ in range(1000)])

In [ ]:
disp = np.random.random(size=(params['NGRIDSAMPLES'], ic.mass.size, 3))
for i in range(0, ic.mass.size):
    for k in range(0, 3):
        y = np.interp(ic.mass[i], mass_tab, disp[:, i, k])

In [ ]:
def create_fname(params):
    '''
    Construct a filename for the output IC based on the parameters.

    Parameters
    ----------
    params : dict
        Dictionary containing the simulation parameters.

    Returns
    -------
    str
        The generated filename.
    '''
    fname = f"{params['IC_PREFIX']}_"
    fname += "Lx{}_Ly{}_Lz{}_".format(*map(int, params['LBOX']))
    fname += f"R3D{params['R_3D']:.0f}_D4D{params['D_4D']:.0f}_z{params['REDSHIFT']:.0f}"
    return fname

In [ ]:
create_fname(params)